In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(12345)

In [ ]:
from PIL import Image, ImageDraw

def make_material_patch(text: str, value: int) -> np.ndarray:
    img = Image.new("L", (64, 64), 0)

    draw = ImageDraw.Draw(img)
    for i in range(0, 64, 12):
        draw.text((0, i), (text + " ") * 15, fill=255)
    a = np.array(img)
    # return a, img
    # print(a.min(), a.max())
    return np.where(a > 0, value, 0).astype(int)

In [ ]:
import NCrystal as NC

xsecs = {}
for name in NC.browseFiles():
    raw = name.name
    if raw == "void.ncmat":
        print("void found: stopping")
        break
    a = raw.removesuffix(".ncmat")
    b = a.split("_")
    for s in b:
        if s.startswith("sg"):
            continue
        if all(x not in s for x in ("Glass", "Bromide", "Carbide", "Nitride", "Gas", "STP", "Epoxy", "Araldite", "Kapton", "Nylon", "Poly", "PEEK", "PVC", "Rubber")):
            new = s
            break
    if "HeavyWater" in new:
        new = "D2O"
    if "LiquidWater" in new:
        new = "H2O"

    # # Load a material (can be element or compound)
    # scatter = NC.createScatter(raw)

    # # Or for a range of energies
    # xs_values = scatter.xsect(ekin=energies)

    # Only keep short names
    if len(new) < 5:
        xsecs[new] = NC.createScatter(raw)


In [ ]:
# xsecs = {"Fe": xsecs["Fe"], "He": xsecs["He"]}

In [ ]:
# shape of grid based on number of materials
nrows = int(np.ceil(np.sqrt(len(xsecs))))
ncols = nrows

materials = {}
i = j = 0
for key, xs in xsecs.items():
    materials[key] = {"loc": (i, j), "xs": xs}
    i += 1
    if i >= nrows:
        i = 0
        j += 1
materials

In [ ]:
nrows, ncols

In [ ]:
len(materials)

In [ ]:
from scipy.interpolate import interp1d

wmin = 0.05
wmax = 15.0

# 1. Pre-compute 1D transmission curves for each material
# E_grid = np.logspace(np.log10(Emin.value), np.log10(Emax.value), 300)

# E_grid = np.logspace(-3, 2, 300)
W_grid = np.linspace(wmin, wmax, 300)
# E_grid = wavelength_to_energy(W_grid).values[::-1]

background = 3.0

thickness = 1.0

# Background (material index 0)
transmission_curves = {
    0: np.exp(-np.full_like(W_grid, fill_value=background) * thickness)
}

# Each material
for i, (mat, data) in enumerate(materials.items()):
    mat_idx = i + 1
    # mu = data["material"].mu(E_grid)
    mu = np.asarray(data["xs"].xsect(wl=W_grid))
    mu = mu / mu.max()  # Normalize to max of 1 for similar intensity across materials
    transmission_curves[mat_idx] = np.exp(-mu * thickness)

# 2. Create interpolators
interpolators = {
    idx: interp1d(W_grid, trans, bounds_error=False, fill_value=0)
    for idx, trans in transmission_curves.items()
}


In [ ]:
from PIL import Image, ImageDraw

def make_material_patch(text: str, dx, pad) -> np.ndarray:
    img = Image.new("L", (dx, dx), 0)

    draw = ImageDraw.Draw(img)
    for i in range(0, 64, 12):
        draw.text((0, i), (text + " ") * 15, fill=255)
    a = np.array(img)
    # return a, img
    # print(a.min(), a.max())
    patch = np.where(a > 0, 1, 0).astype(int)
    out = np.zeros((dx+pad, dx+pad))
    out[pad//2:pad//2+dx, pad//2:pad//2+dx] = patch
    return np.flipud(out)

In [ ]:
dx = 64
pad = 20

# nx = ncols * (dx + pad) + pad
# ny = nrows * (dx + pad) + pad

# material_map = np.zeros((ny, nx), dtype=int)

xx = []
yy = []
ww = []

# Generate events
N = 10_000_000

for i, (mat, data) in enumerate(materials.items()):
    print("Making patch for material", mat)
    row, col = data["loc"]
    # x0 = col * (dx + pad) + pad
    # y0 = row * (dx + pad) + pad

    patch = make_material_patch(mat, dx, pad)

   
    
    x = rng.uniform(0, patch.shape[1], N)
    y = rng.uniform(0, patch.shape[0], N)
    
    # Uniform wavelengths for now
    wav = rng.uniform(wmin, wmax, N)

    # 3. Apply to events
    mat_indices = patch[y.astype(int), x.astype(int)]
    
    # Compute transmission for each event using its material's curve
    transmission = np.zeros(N)

    # 0: background
    mask = mat_indices == 0
    transmission[mask] = interpolators[0](wav[mask])

    # 1: material
    inv = ~mask
    transmission[inv] = interpolators[i+1](wav[inv])
    
    keep = rng.random(N) < transmission

    xx.append(x[keep] + col * (dx + pad))
    yy.append(y[keep] + row * (dx + pad))
    ww.append(wav[keep])
    

# Fill remaining empty patches
for j in range(i, nrows*ncols - 1):
    print("Making background patch", j)

    row += 1
    if row >= nrows:
        row = 0
        col += 1

    x = rng.uniform(0, patch.shape[1], N)
    y = rng.uniform(0, patch.shape[0], N)
    wav = rng.uniform(wmin, wmax, N)

    # Compute transmission for each event using its material's curve
    transmission = np.zeros(N)

    # 0: background
    transmission = interpolators[0](wav)
    
    keep = rng.random(N) < transmission

    xx.append(x[keep] + col * (dx + pad))
    yy.append(y[keep] + row * (dx + pad))
    ww.append(wav[keep])

In [ ]:
nevents = sum(len(a) for a in xx)
nevents

In [ ]:
import scipp as sc

events = sc.DataArray(
    data=sc.ones(sizes={"event": nevents}),
    coords={
        "x": sc.array(dims=["event"], values=np.concatenate(xx), unit="mm"),
        "y": sc.array(dims=["event"], values=np.concatenate(yy), unit="mm"),
        "wavelength": sc.array(dims=["event"], values=np.concatenate(ww), unit="meV"),
    }
)

# events.coords['wavelength'] = energy_to_wavelength(events.coords['E'])
# events.coords['energy'] = wavelength_to_energy(events.coords['wavelength'])


events

In [ ]:
%matplotlib widget

In [ ]:
events.hist(y=256, x=256).plot()

In [ ]:
Fe = events.bin(x=sc.array(dims=["x"], values=[170., 250.], unit="mm"),
                y=sc.array(dims=["y"], values=[260., 330.], unit="mm"))
Fe

In [ ]:
Fe.squeeze().hist(wavelength=300).plot()

In [ ]:
import plopp as pp

pp.scatter(events[::400])

In [ ]:
binned = events.bin(y=10, x=10)
binned

In [ ]:
binned.hist().plot()

In [ ]:
a = binned['x', 0]['y', 0]
a

In [ ]:
a.hist(y=256, x=256).plot()